# 00 — 베이스 모델 선정 (3종 zero-shot F1)

A/B/C가 **동일 베이스**에서 출발하도록 베이스를 먼저 확정한다(스펙 §2).
후보: **Qwen3-1.7B**, **Qwen3-4B**, **Llama-3.2-3B-Instruct**.

> ## ⚙️ 실행 모드 배너 — 이 노트북은 **CPU 스모크**로 실행됨
>
> **왜:** 대상 Azure 구독(`ME-MngEnvMCAP756842-hjeon-1`)에 **모던 GPU 쿼터가 0**이고,
> H100·A100·A10 **쿼터 증설 요청을 제출**했으나 승인되지 않았습니다.
> (요청 IDs — H100 `e9245f08`: Failed, A10 `d9addbec`: Failed, A100 `bd193591`: InProgress.
> 과거 동일 패밀리 요청도 모두 Failed — 스폰서 구독의 GPU 잠금으로 판단.)
>
> **결과:** "쿼터 중 있는 걸로 처리"하라는 지시에 따라, 파이프라인을 **동일 코드 경로**로
> 소형 모델(`Qwen/Qwen2.5-0.5B-Instruct`) + 200개 서브셋 + 12스텝의 **CPU 스모크**로
> 실제 실행해 모든 블록이 통과함을 **실 출력**으로 증명합니다.
>
> **프로덕션(스펙) 실행:** GPU VM에서 `config.yaml`의 `compute.mode: gpu`(base=`Qwen/Qwen3-1.7B`,
> `train.backend: unsloth`, BF16, full data)로 **동일 셀**을 실행하면 스펙의 A100/H100 BF16
> 베이스라인이 나옵니다. 코드는 그대로, 설정만 다릅니다.

In [1]:
import os, sys
here = os.getcwd()
for cand in [here, os.path.dirname(here), os.path.join(here, "pdf_qa_extraction"),
             os.path.dirname(os.path.dirname(here))]:
    if os.path.isdir(os.path.join(cand, "quantization")):
        if cand not in sys.path:
            sys.path.insert(0, cand)
        os.chdir(cand)
        break
print("cwd:", os.getcwd())

cwd: /ai-work/copilot/PDF2LLM-Tuning-Studio/pdf_qa_extraction


In [2]:
from quantization.data_korquad import load_config, load_korquad
cfg = load_config(force_mode='cpu')  # GPU VM: load_config()
cfg['compute']['mode'], cfg['base_model']['selected']

('cpu', 'Qwen/Qwen2.5-0.5B-Instruct')

### 후보 3종

In [3]:
candidates = cfg['base_model']['candidates']
print('후보 3종 (스펙 §2):')
for c in candidates:
    print(f"  - {c['id']:34} family={c['family']:6} gated={c['gated']}")
print('\n선정 기준: (a) KorQuAD dev zero-shot F1, (b) 단일 GPU 적합, (c) TorchAO INT4+vLLM 호환')

후보 3종 (스펙 §2):
  - Qwen/Qwen3-1.7B                    family=qwen   gated=False
  - Qwen/Qwen3-4B                      family=qwen   gated=False
  - meta-llama/Llama-3.2-3B-Instruct   family=llama  gated=True

선정 기준: (a) KorQuAD dev zero-shot F1, (b) 단일 GPU 적합, (c) TorchAO INT4+vLLM 호환


### zero-shot F1 측정 하네스
학습 없이 base를 로드 → held-out KorQuAD → EM/F1. (CPU 스모크는 소형 프록시 1종만 실측.)

In [4]:
# zero-shot F1 하네스: 각 후보를 (학습 없이) 로드 -> held-out eval -> F1.
# GPU VM에선 candidates 3종 전부 루프. CPU 스모크에선 시간/게이팅 때문에 소형 모델 1종만 실증.
import quantization.eval_qa as E
data = load_korquad(cfg)
rows = []
smoke = cfg['compute']['mode'] == 'cpu'
run_list = ([{'id': cfg['base_model']['smoke'], 'family': 'qwen', 'gated': False, 'note': 'CPU-smoke proxy'}]
            if smoke else cfg['base_model']['candidates'])
for c in run_list:
    if c.get('gated'):
        rows.append({'model': c['id'], 'f1': None, 'note': 'gated: HF 승인+토큰 필요(VM에서 측정)'})
        continue
    model, tok = E.load_model_for_eval(c['id'], 'fp32' if smoke else 'bf16')
    r = E.evaluate_model(model, tok, data['eval'], method='zeroshot', base_model=c['id'],
                         max_new_tokens=cfg['eval']['max_new_tokens'],
                         batch_size=cfg['eval']['batch_size'], ppl_samples=0)
    rows.append({'model': c['id'], 'em': r.exact_match, 'f1': r.f1,
                 'note': c.get('note', 'zero-shot')})
    del model
rows

/home/dev/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[{'model': 'Qwen/Qwen2.5-0.5B-Instruct',
  'em': 0.0,
  'f1': 17.961,
  'note': 'CPU-smoke proxy'}]

### 선정

In [5]:
# 선정: zero-shot F1 최고 + 단일 GPU 적합 + INT4/vLLM 호환. (CPU 스모크는 프록시 1종이라
# 선정은 스펙 기준의 기본값 Qwen/Qwen3-1.7B를 config.yaml에 고정; GPU에서 위 표로 확정.)
print('현재 config 선정 base:', cfg['base_model']['selected'])
print('CPU 스모크에선 3-way 실측이 불가(4B는 CPU 과도, Llama-3.2-3B는 gated).')
print('=> GPU VM에서 위 셀을 3종 전부로 실행해 최종 F1 표로 확정하세요.')

현재 config 선정 base: Qwen/Qwen2.5-0.5B-Instruct
CPU 스모크에선 3-way 실측이 불가(4B는 CPU 과도, Llama-3.2-3B는 gated).
=> GPU VM에서 위 셀을 3종 전부로 실행해 최종 F1 표로 확정하세요.
